# 03 · Embeddings：把文字变成向量

关键词不完全相同，仍然可能表达同一个意思：比如“面料成分”和“材质”。Embedding 模型把文字变成一串数字，让程序可以计算语义接近程度。

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
base_url = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
client = OpenAI(api_key=api_key, base_url=base_url or None) if api_key else None
print("API 已配置" if client else "未配置 API：保留本地步骤，调用模型的单元会跳过")

import numpy as np

def embed(texts):
    if not client:
        raise RuntimeError("请先配置 Embedding API 的密钥。")
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in response.data], dtype=float)

未配置 API：保留本地步骤，调用模型的单元会跳过


## 生成向量

In [2]:
texts = [
    "瑜伽裤使用 Nylon 66 和 Lycra",
    "冲锋衣采用三层防水面料",
    "女款腰围尺码对照表",
]

if client:
    vectors = embed(texts)
    print("文本数量：", len(vectors))
    print("向量维度：", vectors.shape[1])
else:
    print("未调用 Embedding API：请配置 API 后运行本单元")

未调用 Embedding API：请配置 API 后运行本单元


## 用余弦相似度比较语义

先把向量归一化，再用点积排序。分数越高，表示两个文本在这个模型看来越接近。

In [3]:
if client:
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    question = "瑜伽裤的材质是什么？"
    question_vector = embed([question])[0]
    question_vector = question_vector / np.linalg.norm(question_vector)
    scores = vectors @ question_vector
    for text, score in sorted(zip(texts, scores), key=lambda item: item[1], reverse=True):
        print(f"{score:.3f}  {text}")